## Vorlage zur Auswertung des Myon-Versuchs
## $~~~~$  "Nachweis der Präzession des Myonenspins im Magnetfeld"

Dieses Notebook baut auf dem Notebook `lifetime.ipynb` zur Messung der Myon-Lebendauer auf.
Sie können es entweder erweitern oder diese Vorlage zur Auswertung der Langzeitmessung
verwenden und benötigte Code-Teile übertragen.

Auch mit den über einen Zeitraum von einer Woche aufgenommenen Daten ist keine genaue Messung 
der Größe des Myon-Spins möglich. Nebenmimia im Parameterraum erschweren die Anpassung zusätzlich.
Außerdem ist die Anpassung eines exponentiell gedämpften Oszillationssignals an Daten mit 
statistischen Fluktuationen nicht eindeutig, weil dabei zufällige Periodizitäten auftreten, die
leicht das eigentliche Signal überdecken können. 

Stattdessen wird zur Auswertung ein Verfahren verwendet, wie es auch in Suchen nach Abweichungen 
von einer Standardtheorie (sog. "neuer Physik") üblich ist. 
Im konkreten Fall fassen wir die kleine erwartete Modulation der Lebensdauerverteilung als
Abweichung von einem rein exponentiellen Verhalten auf, das wir als "Null-Hypothese" im Sinne eines
Hypothsentests ansehen. Dazu werden alle Parameter des Modells mit Spin-Modulation der Lebensdauer
festgehalten und nur die Amplitude der Spin-Modulation in der Anpassung frei gelassen. Für eine
Modulationsamplitude von Null entspricht das Modell dann der Null-Hypothese. Eine signifikante 
Abweichung des aus der Anpassung bestimmten Werts der Modulationsamplitude wird als Evidenz für 
die Existenz des Myon-Spins gewertet. 

Wie deutlich das Signal zu sehen ist, wird als "Signifikanz" ausgedrückt. In der Statistik ist 
das die a-posteriori Wahrscheinlichkeit dafür, eine noch kleinere Übereinstimmung zwischen angepasstem
Modell und der Null-Hpyothese zu erhalten als tatsächlich beobachtet. Diese Information lässt sich
vollständig aus der Likelihood-Fuktion (besser der Profil-Likelihood) für verschiedene Werte der
Modulationsamplitude bestimmen. Hintergurndinformation finden Sie im Kapitel zur Statistik im 
"Blauen Buch" oder in den jupyter-Tutorials *negLogLFits.ipynb* und *advancedFitting.ipynb*
unter der URL https://etpwww.etp.kit.edu/~quast/jupyter/ .

### Verfahren zum Nachweis des Myon-Spins

Als Alternative zum einfachen exponentiellen Zerfallsgesetz formulieren wir eine Parametriesierung
der Anzahl der zur Zeit $t$ im oberen Detektor nachgewiesenen Zerfälle, die einen (kleinen) zusätzlichen
Anteil enthält (s. Versuchsanleitung im Blauen Buch):

  $${\rm pdf}_{\rm signal} = 
    K \cdot e^{-\frac{t}{\tau}} \cdot \left[1 + \bar{A}\cdot \cos{(\omega t + \delta)}\right]$$

Damit wir die entspechende Wahrscheinlichkeitsdichte in der Anpassung verwenden können, muss sie
im sensitiven Zeitintervall $t\in[a,b]$ normiert werden. Das dazu notwendige Integral der unnormierten
Verteilung bestimmt man am einfachsten durch Verwendung eines Computer-Algebra-Systems. 

Das Ergebnis lautet:

  $$ K^{-1}(\tau, \bar{A}, \omega, \delta, a, b) = 
  {\tau  \left(\frac{\bar{A} \left(e^{-\frac{a}{\tau}} (\cos (a \omega +\delta ) - \tau \omega \sin (a   \omega +\delta ))+e^{-\frac{b}{\tau}} (\tau  \omega  \sin (b \omega +\delta )-\cos (b \omega +\delta   ))\right)}{\tau ^2 \omega ^2+1}+e^{-\frac{a}{\tau }}-e^{-\frac{b}{\tau }}\right)}$$

Die gesamte Wahrscheinlichkeitsdichte ist eine Summe aus der erwarteten Signalverteilung und dem
(konstanten) Untergrund: 

  $$ {\rm pdf}_\rm{bkg}(a, b) = \frac{1}{b-a} $$

Mit einem Untergrundanteil $f$ ergibt sich insgesamt: 
 
 $$ {\rm pdf} (t; \tau, f, \bar{A}, \omega, \delta, a, b) = 
      (1-f) \cdot {\rm pdf}_{\rm signal}   
     + f \cdot {\rm pdf}_{\rm bkg} $$ 
 
Bei geeigneter Wahl der Parameter gilt diese Formel für Zerfälle, deren Zerfallselektron im oberen
Detektor nachgewiesen wird. Für Elektronen im Detektor unter dem Kupferabsorber ist die Modulation
um 180° phasenverschoben - durch entsprechende Wahl des Parameters $\delta$ ist der gleiche Code 
dann ebenfalls für den unteren Detektor verwendbar.                                   

### *Pyhton*-Code für die Wahrscheinlichkeitsdichten

Die folgende Code-Zelle stell den notwendigen *Python-Code* für die Wahrscheinlichkeitsdichten bereit: 

In [1]:
import numpy as np

# set valid range of measured lifetimes (maximum values given here, may not be optimal)
a0=0.6e-6
b0=15.e-6


# PDFs for exponential decay and modulation by spin rotations

def exp_pdf(t, tau=2.19e-6, fbg=0.1, a=a0, b=b0):
    """
    Probability density function for the decay time of a myon using the Kamiokanne-Experiment.
    The pdf is normed for the interval (a, b).

    :param t: decay time
    :param fbg: background
    :param tau: expected mean of the decay time
    :param a: the minimum decay time which can be measured
    :param b: the maximum decay time which can be measured
    :return: probability for decay time x
    """
    pdf1 = np.exp(-t / tau) / tau / (np.exp(-a / tau) - np.exp(-b / tau))
    pdf2 = 1. / (b - a)
    return (1 - fbg) * pdf1 + fbg * pdf2

# set some initial values
tau0=2.19e-6
omega0=3.5e6
delta0=0. 
Abar_top0=0.05
f_top0=0.5
Abar_bot0=0.05
f_bot0=0.5

def spin_pdf(t, tau=2.19e-6, fbg=0.3, a_bar = 0.2, delta=0, omega=3.5e6, a = a0, b = b0):
    """
    Probability density function for the decay time of a myon using the "Properties of Cosmic Muons"-experiment in the
    physics master advanced laboratory course (with applied magnetig field)
    The pdf is normed for the interval (a, b).

    :param t: decay time
    :param tau: expected mean of the decay time
    :param fbg: percentage of background (assumed to be flat) 
    :param a_bar: percentage of the modulation due to spin precession
    :param delta: phase shift between spin polarization and electron/positron impulse (should be 0 or pi)
    :param omega: precession angular frequency
    :param a: the minimum decay time which can be measured
    :param b: the maximum decay time which can be measured
    :return: probability for decay time x
    """
    modulation_pdf = np.exp(-t/tau)*(1 + a_bar*np.cos(delta + t*omega))
    modulation_pdf_integral = tau*(np.exp(-a/tau) - np.exp(-b/tau) + a_bar*(np.exp(-a/tau) * (np.cos(delta + a*omega) - tau*omega*np.sin(delta + a*omega)) - np.exp(-b/tau) * (np.cos(delta + b*omega) - tau*omega*np.sin(delta + b*omega))) / (1 + tau**2 * omega**2))
    bg_pdf = 1/(b - a)
    full_pdf = (1-fbg) * modulation_pdf/modulation_pdf_integral + fbg*bg_pdf
    return full_pdf

def spin_pdf_top(t, tau=tau0, Abar_top=Abar_top0, omega=omega0, delta=delta0, f_top=f_top0, a=a0, b=b0):
    # for top detector: Abar = Abar_top
    return spin_pdf(t, tau=tau, a_bar=Abar_top, omega=omega, delta=delta, fbg = f_top, a=a, b=b)

# pdf for decay election in bottom detectors
def spin_pdf_bot(t, tau=tau0, Abar_bot=Abar_bot0, omega=omega0, delta=delta0, f_bot=f_bot0, a=a0, b=b0):
    # for bottom detector: delta_bot = delta_top+pi
    return spin_pdf(t, tau=tau, a_bar= Abar_bot, omega=omega, delta=delta+np.pi, fbg = f_bot, a=a, b=b)

# check proper normalization of pdfs
import scipy.integrate as integrate
print("** check normalisation of PDFs (must be 1. !) :")
print('  normalisation exp_pdf_top():', integrate.quad(exp_pdf, a0, b0))
print('  normalisation spin_pdf_top():', integrate.quad(spin_pdf_top, a0, b0))
print('  normalisation spin_pdf_bot():', integrate.quad(spin_pdf_bot, a0, b0))

** check normalisation of PDFs (must be 1. !) :
  normalisation exp_pdf_top(): (1.0000000000000004, 1.110223024625157e-14)
  normalisation spin_pdf_top(): (0.9999999999999998, 2.4204410905979057e-13)
  normalisation spin_pdf_bot(): (1.0, 2.377625951041618e-13)


### Aufgabe: Grafische Darstellung des erwarteten Signals

Um ein Gefühl für die Größe des Modulationssignals zu bekommen, sollten Sie die Verteilung zunächst einmal grafisch darstellen - 
damit überprüfen sie auch die Korrektheit der Implementierung der Funktionen in *Python*. Außerdem ist es wichtig, für die 
spätere Analyse die passsende Anzahl an Bins in den Histogrammen zu wählen. 

Zunächst muss dazu die erwartete Oszillationsfrequenz $\omega_\mathrm{lit}$ aus dem gemessenen Magnetfeld $B$ zu berechnet werden.

Ferner wird der anzupassende Bereich für die Wahrscheinlichkeitsdichte bestimmt: Auf der einen Seite werden zu kurze Zerfallszeiten $t \lesssim 0.6 \mathrm{\mu s}$ herausgefiltert, da sie hauptsächlich durch Kerneinfänge der $\mu^-$ im Kupfer, mit weit kürzeren Zerfallszeiten als die Lebensdauer des Myons, dominiert werden. Auf der anderen Seite wird die obere Grenze durch die maximale Messzeit der Versuchsapparatur bestimmt, es können $t \gtrsim 14.5 \mathrm{\mu s}$ herausgefiltert werden. Mit den genauen Werten kann etwas herumexperimentiert werden.

Für den Untergrundanteil wird geschätzt, dass $f_\mathrm{bg} \approx 30\%$ ist. 
Für die Stärke der Präzessionsmodulation wird ein sehr gut sichtbarer Anteil von $\bar{A} \approx 15\%$
gewählt; in der Realität beträgt dieser Anteil nur wenige Prozent !
Die Phasenverschiebung $\delta$ ergibt sich aus der Zerfallskinematik und der Polarisation der eintreffenden Myonen. Physikalisch sinnvoll sind dabei nur zwei Werte 

$\delta = \begin{cases} 0 & \textrm{oberer Detektor} \\ \pi & \textrm{unterer Detektor} \end{cases}$.

Leicht von Null bzw. $\pi$ verschiedene Werte ergeben sich durch Zeitverzögerungen in der Signalverarbeitungskette relativ zum Myon-Durchgang. 

Die Lebensdauer des Myons wird auf den Literaturwert $\tau \approx 2.19\mathrm{\mu s}$ gesetzt.

Es bietet sich an, mögliche Grenzen für die Häufigkeitsklassen (=Bins) eines Histogramms gleich
mit in das Diagramm zu zeichnen. Einererseits werden mehr Bins benötigt um die Modulation durch
die Präzession besser aufzulösen, aber andererseits führen zu viele Bins zu hohen statistischen
Unsicherheiten.

Eine Vorlage zur Lösung dieser Aufgabe finden sie in der folgenen Code-Zelle:

In [2]:
import scipy.constants as c
import matplotlib.pyplot as plt

# Das erwartete omega wird aus dem gemessenen B-Feld berechnet
g_lit = np.abs(c.physical_constants["muon g factor"][0])
e = c.physical_constants["elementary charge"][0]
m = c.physical_constants["muon mass"][0]
B = #TODO: Berechnetes B-Feld (in Tesla)
omega_lit = #TODO: Formel zur berechnung von omega einfügen
print(f"Das erwartete omega ist: {omega_lit:.2e}")

# Den gezeichneten Bereich auf geeignete Werte festlegen
limits = (0.6e-6, 0.00e-6)  #TODO: Hier eine geeignete untere und obere Grenze einfügen 'limits = (untere_grenze, obere_grenze)'
t = np.linspace(limits[0], limits[1], 1000)
lande_top = pdf(t, tau=2.19e-6, fbg=0.30, a_bar=0.15, delta=0, omega=omega_lit, a=limits[0], b=limits[1])
lande_bot = pdf(t, tau=2.19e-6, fbg=0.30, a_bar=0.15, delta=np.pi, omega=omega_lit, a=limits[0], b=limits[1])

plt.subplots(figsize=(10,7))
plt.title("Erwartete Wahrscheinlichkeitsdichte in den Detektoren")
plt.plot(t*1e6, lande_top, label="Zerfälle nach oben")
plt.plot(t*1e6, lande_bot, label="Zerfälle nach unten")
#TODO: Einen geeigneten Wert für die Anzahl der Häufigkeitsklassen im Histogramm später finden
for x in np.linspace(limits[0], limits[1], 90)*1e6:
    #                                       ^--- Anzahl Häufigkeitsklassen
    plt.axvline(x, color="black", linestyle=":", alpha=0.3)
plt.legend(loc="best")
plt.ylabel("Dichte")
plt.xlabel(r"$\mathrm{\mu^+}$ Lebensdauer in $\mathrm{\mu s}$")
plt.show()

SyntaxError: invalid syntax (1383214622.py, line 8)

### Aufgabe: Einlesen der Messdaten und Qualitätskontrolle

Im nächsten Schritt werden die gemessenen Daten eingelesen. Wie schon im Versuchsteil 
"Messung der Myon-Lebensdauer" ist es bei den Daten der Lanzteitmessung noch wichtiger,
sich zunächst von der Qualität der (unbeaufsichtigt) aufgezeichneten Daten zu überzeugen
und ggf. den Datensatz zu bereinigen, wenn es Unterbrechungen der Datennahme, Perioden 
mit hohen Rauschraten oder ander Unregelmäßigkeiten gab. 

Als erstes empfiehlt es sich, zunächst die Qualität der Daten zu überpüfen:

  -  statistische Information zur Datennahme
  -  zeitlicher Verlauf der aufgenommenen Daten, d.h. die in Zetiintervallen von einigen Minuten 
      beobachtete Anzahl an Ereignissen 
  -  Pulshöhenverteilungen in den Detektoren für jeweils erste und verzögerte Pulse.

Als Vorlage können Sie den entprechenden Code aus dem notebook *lifetime.ipynb* und die dort
definierte Funktion *dataquality_check()* verwenden. 

In [ ]:
# -->> insert your code here  <<--

### Aufgabe: Filtern der Messdaten

Als nächstes müssen für die geplante Datenauswertung geeignete Ereignisse ausgewählt werden. 
Die Myonen sollen möglichst in der Kupferplatte im Magnetfeld gestoppt worden sein, und
Ereignisse mit klar nachgewiesenen Positronen aus dem Myon-Zerfall sollten selektiert werden.
Dabei muss zwischen Ereignissen mit nach oben bzw. nach unten emittierten Positronen unterschieden
werden. 

Information über die Modulation der Exponentialverteilung stecken sowohl in den Daten mit
Zerfallspositronen im oberen als auch im unteren Detektor. Allerdigs ist die Sigifikanz 
wegen des günstigeren Verhältnisses aus Signal zu Untergund im oberen Detektor höher. 
Es ist also ausreichend, die Auswertung zunächst für die oberen Detektoren auszuführen. 

Die auszuwählende Signatur für die oberen Detektorlagen besteht aus einer Koinzidenz der 
ersten Pulse in den oberen Detetorlagen A und B und keinem Puls in den Detektoren C 
(und auch nicht in D, falls dieser ebenfalls ausgelesen wurde). 
Verzögerte zweite  Pulse darf es nur in den Detektorlagen A und B geben. 

Die Signatur für Zerfälle mit nach unten emittiertem Zerfallselektron ist für die ersten Pulse identisch; 
verzögerte Pulse darf es in diesem Fall nur in den Detektorlagen C oder D geben. 

Die Schwellen für die Pulshöhen sollten so angepasst werden, dass einerseits möglichst viel 
Signalanteil und möglichst wenig Untergrund enthalten ist. Andererseits muss die Gesamtzahl 
der Ereignisse noch ausreichen, um die Wahrscheinlichkeitsdichten gut anpassen zu können. 
Nach einer ersten Anpassung der Wahrscheinlichkeitsdichte sollen diese Schnitte iterativ 
abgeändert werden, bis ein möglichst optimales Ergebnis erreicht wurde. Die Zielgröße bei 
dieser händischen Optimierung ist die relative Unsicherheit des Parameters $\bar{A}$.

Als Vorlage können Sie den Code aus dem Notebook *lifetime.ipynb* und die dort definierte
Funktion *select()* verwenden. 

In [ ]:
# -->> insert your code here  <<--

import pandas as pd

# top_selected = ...


# bot_selected = ...


### Aufgabe: Anpassungen des rein exponentiellen Modells ("Null-Hypothese") und des Modells mit Spin-Modulation
Nach der Datenselektion wird der Rest der Auswertung ausschließlich an Hand der nun in den Arrays Lt_top (bzw. lt_bot) 
enthaltenen gemessenen Lebensdauern ausgeführt.

Die gemessenen Zerfallszeiten werden in ein Histogramm eingetragen; zunächst wird die Wahrscheinlichkeitsdichte ohne 
Präzessionsmodulation an die histogrammierten Daten angepasst. Es ist also $\bar{A} = 0, \, \omega = 0, \, \delta = 0$. 
Die Parameter $a$ und $b$ werden auf die oben bestimmten Grenzen festgelegt und der Untergrundanteil wird auf das 
physikalisch sinnvolle Intervall $f_\mathrm{bg} \in [0, 1]$ eingeschränkt. 

Man erhält durch die Anpassung Werte für $\tau$ und $f_\mathrm{bg}$, die in einer zweiten Anpassung als fixierte Parameter 
für die Wahrscheinlichkeitsdichte mit Präzessionsmodulation verwendet werden. Ferner werden nun $\omega$ und $\delta$ auf 
die oben berechneten bzw. beschriebenen Werte festgelegt. Es findet also ausschließlich eine Anpassung des $\bar{A}$-Parameters 
statt, wobei auch hier auf das physikalisch sinnvolle Intervall $\bar{A} \in [0, 1]$ eingeschränkt werden muss.

Die Anpassungen an die Histogramme können Sie ganz analog zu den Beispielen aus dem Notebook `lifetime.ipynb` wahlweise mit 
den Wrapper-Funktionen in *PhyPraKit* (*hFit* bzw. *k2hFit* für *kafe2*) oder direkt mit den Klassen und Methoden von *kafe2* 
durchführen.

### Aufgabe: statistisch Auswertung und Bestimmung der Signal-Signifikanz 

Ob eine signifikante Oszillation beobachtet wurde, entscheidet sich anhand der Werte des negativen 
Logarithmus der Likelihood-Funktion, die die Fits als Ergebnis ausgeben. 
Als letzter Schritt werden also mit Hilfe der gefundenen Funktionsparameter die Likelihood-Werte 
mit und ohne Präzessionsmodulation miteinander verglichen. 
Als Teststatistik wird dafür der negative Logarithmus des Likelihood-Verhältnises verwendet
(siehe Statistik-Kapitel im blauen Buch, das Verfahren basiert auf dem sog. "Neyman-Pearson-Lemma"):

$$ z^2 = -2 \cdot nl\mathcal{L} = -2\cdot \left(
    \displaystyle\sum_{i=1}^{n_d} \, -\ln\left(\mathrm{pdf_1}(x_i, \vec p)\right) \,
    - \displaystyle\sum_{i=1}^{n_d} \, -\ln\left(\mathrm{pdf_2}(x_i, \vec p)\right)
     \right) $$
     
Aus dieser Differenz ergibt sich direkt das Quadrat der Signifikanz (der sogenante $z$-Wert) 
für den Parameter $\bar{A}$, d.h. anschaulich den Abstand von Null gemessen in Standardabweichungen. 
Eine etwas ungenauere Bestimmung erhalten Sie auch, wenn Sie den Wert von $\bar{A}$ aus der Anpassung
durch seine Unsicherheit teilen, 
$$ z \simeq \hat{\bar{A}} / \sigma_\bar{A}\,.$$ 
Diese anschauliche Größe drück aus, wie weit das Ergebnis, gemessen in Einheiten der Unsicherheit, 
von Null abweicht. 

Mit Hilfe der kumulierten Verteilungsfuktion $\Phi$ der Standard-Normalverteilung kann $z$ für
positive Werte von $z$ in einen in der professionellen Statistik üblichen $p$-Wert umgerechnet 
werden: 

$$ p = 1 - 2\,(1 - \Phi(\mathrm{z}; \, \mu = 0, \sigma = 1)), \, z\ge 0\,.$$

Der $p$-Wert ist die a-posteriori Wahrscheinlichkeit dafür, dass das beobachtet Signal auf eine
statitische Fluktuation der Null-Hypothese zurückzuführen ist. Wenn diese Wahrscheinlichkeit
zu klein ist, wird die Null-Hpyothese zu Gunsten der Alternative verworfen. Dabei entspricht
$z=0$ $p=50\%$, $z=1$ entspricht $p=13.6\%$, $z=2$ entspricht $p=2.1\%$ und $z=3$ entspicht 
$p=0.13\%$.

Zur Bestimmung von $z$ können Sie direkt die von den Anpassungen ausgegebenen Werte 
für *gof* =  "goodnes of fit" oder  $-2 \cdot nl\mathcal{L}$ zurück greifen. Diese 
Größen sind lediglich geeignet normierte Werte des negativen Logarithmus der 
Likelihood-Funkton (*2nlL*). Bei der Bildung des Likelihood-Verhältnisses (bzw. der
Subtraktion der *2nlL*-Werte) der beiden Anpassungen fällt diese Normierung wieder heraus.

Sie können $z^2$ auch direkt durch Einsetzen der Daten für die Werte der angepassten Parameter 
in die oben gegebene Formel bestimmen. 

Am Ende sollten Sie zusammenfassend das Ergebnis zusammen stellen

In [ ]:
from scipy.stats import chi2, norm
# Parameter aus dem letzten Fit übernehmen
a = #TODO
b = 
tau = 
fbg = 
a_bar = 
delta = 
omega = 

# Negative log-Likelihood für beiden
non_modulated_2nlL = 
modulated_2nlL = 
z = 

# Teststatistik (-2 * negative log-Likelihood Differenz) berechnen
Delta =
# Ist der Unterschied zwischen den Modellen genau ein Parameter, kann der z-score berechnet werden
z_score = np.sqrt(Delta)
print("Negative Log Likelihood:")
print("\tnon-modulated: {:.2f}".format(non_modulated_log_likelihood))
print("\t    modulated: {:.2f}".format(modulated_log_likelihood))
print("\t        Delta: {:f}".format(Delta))
print("\t      z-score: {:.2f}".format(z_score))
print("\t            p: {:.2%}".format( 1 - norm.cdf(z_score) ) 


### Aufabe: Kommentieren Sie Ihr Ergebnis

### Optional:  Messung des Landé-Faktors ("g-Faktor") des Myons

Ob Sie eine hohe Signifikanz erhalten haben, hängt einersteits von statistischen Fluktuationen
der Daten und auch von Ihren Auswahlkriterien ab.  Wenn Sie sowohl im oberen als auch im unteren
Detektor $z$-Werte von mehr als Drei erhalten haben, können Sie eine Anpassung der Präzessionsfrequenz
versuchen. Um die Daten optimal auszunutzen, sollte eine simultane Anpassung der Daten an die
beiden Lebensdauerverteilungen mit Zerfallspositronen im oberen und im unteren Detektor durchgeführt
werden. Das gelingt mit der in *kafe2* implementierten Klasse "MultiFit", mit der Sie Fits an die
top- und bottom-Detektoren kombinieren können. In beiden Modellen vorhandene Parameter, d.h. die 
mittlere Myon-Lebensdauer $\tau$ und die Präzessionsfrequenz $\omega$, werden in beiden  
Modellfunktionen gleich gesetzt, während die übrigen Parameter jeweils unabhängig behandelt
werden. 

Zuverlässig gelingt solch eine Anpassung, wenn man die über mehrere Wochen aufgenommenen Daten
zusammenfasst.